# Tutorial 2: Virus Phylogenetic Tree Monophyly Analysis

**Detect the phylogenetic relationships of viral taxonomic groups with PyiTOL**

**病毒系统发育树单系性分析——使用 PyiTOL 检测分类群的系统发育关系**

This notebook demonstrates how to use PyiTOL's monophyly detection module to classify
taxonomic groups as monophyletic, paraphyletic, or polyphyletic. We will:

1. Generate a synthetic 30-leaf virus phylogenetic tree with known group structure
2. Create taxonomy data with Genotype and Clade columns
3. Run `check_monophyly()` to classify each group
4. Interpret the three-way classification results
5. Generate a color-strip template with monophyly-based coloring

本教程展示如何使用 PyiTOL 的单系性检测模块，将分类群判定为单系群、并系群或多系群。我们将：

1. 生成一棵具有已知类群结构、包含 30 个末端（叶）的合成病毒系统发育树
2. 创建带有 Genotype 和 Clade 两列的分类学数据
3. 运行 `check_monophyly()` 对每个类群进行判定
4. 解读三分法的判定结果
5. 生成按单系性状态着色的色带模板

---

**Background / 背景知识:**

| Term | Definition |
|------|------------|
| **Monophyletic**（单系群） | All members share a common ancestor, and ALL descendants of that ancestor are included（所有成员共享一个共同祖先，并且该祖先的全部后代都被纳入该群） |
| **Paraphyletic**（并系群） | All members share a common ancestor, but SOME descendants are excluded（所有成员共享一个共同祖先，但有一部分后代被排除在外） |
| **Polyphyletic**（多系群） | Members are scattered across multiple independent clades（成员分散在多个彼此独立的分支上） |

## 1. Setup and Imports

Import the required libraries.

导入所需的库。

In [ ]:
import random
import tempfile
from pathlib import Path

import dendropy
import pandas as pd

from pyitol.core.monophyly import check_monophyly
from pyitol.templates.generator import (
    generate_color_strip_template,
    generate_branch_template,
)
from pyitol.templates.presets.nature import get_nature_palette

## 2. Build a Synthetic Virus Tree with Known Monophyly Structure

We construct a tree where we **know** which groups should be monophyletic, paraphyletic,
or polyphyletic. This lets us verify the algorithm's output.

The tree has 30 leaves organized as follows:

- **Genotype A** (8 leaves): monophyletic -- all in one clade (`Clade_I`)
- **Genotype B** (8 leaves): monophyletic -- all in one clade (`Clade_III`)
- **Genotype C** (6 leaves): not monophyletic -- C_01-C_03 sit in `Clade_II`, while
  C_04-C_06 sit in `Clade_IV` (where C_06 is sister to D_06-D_08)
- **Genotype D** (8 leaves): not monophyletic -- D_01-D_05 sit in `Clade_II`, which also
  contains some Genotype C members, while D_06-D_08 sit in `Clade_IV`

我们构建一棵树，在其中**事先知道**各个类群应当属于单系群、并系群还是多系群，
这样就能验证算法的输出。

该树共有 30 个末端（叶），组织方式如下：

- **Genotype A**：8 个末端（叶），单系群——全部位于同一个分支内（`Clade_I`）
- **Genotype B**：8 个末端（叶），单系群——全部位于同一个分支内（`Clade_III`）
- **Genotype C**：6 个末端（叶），非单系群——C_01-C_03 位于 `Clade_II`，而 C_04-C_06 位于
  `Clade_IV`（在 `Clade_IV` 中 C_06 与 D_06-D_08 互为姐妹群）
- **Genotype D**：8 个末端（叶），非单系群——D_01-D_05 位于 `Clade_II`（该分支中还含有部分
  Genotype C 的成员），而 D_06-D_08 位于 `Clade_IV`

In [ ]:
# We build the tree from a Newick string that encodes our desired topology.
# The root is an unresolved polytomy with 4 child clades:
#   root
#   ├── Clade1 (Genotype A: A_01..A_08) -- monophyletic
#   ├── Clade2
#   │   ├── SubClade2a (Genotype D: D_01..D_05)
#   │   └── SubClade2b (Genotype C: C_01..C_03)
#   ├── Clade3 (Genotype B: B_01..B_08) -- monophyletic
#   └── Clade4
#       ├── SubClade4a (Genotype C: C_04, C_05)
#       └── SubClade4b (Genotype C: C_06 sister to Genotype D: D_06..D_08)
# Neither C nor D is monophyletic: each occupies two of the four root child
# clades (C_01..C_03 in Clade2 plus C_04..C_06 in Clade4; D_01..D_05 in Clade2
# plus D_06..D_08 in Clade4).  Because those members meet at the unresolved
# root polytomy, check_monophyly() with the default polytomy_mode='strict'
# labels them 'data_insufficient' instead of giving a definitive
# paraphyletic/polyphyletic call (see section 4.2).

newick = (
    "((((A_01:0.1,A_02:0.1):0.1,(A_03:0.1,A_04:0.1):0.1):0.1,"
    "((A_05:0.1,A_06:0.1):0.1,(A_07:0.1,A_08:0.1):0.1):0.1):0.2,"
    "(((D_01:0.1,D_02:0.1):0.1,(D_03:0.1,(D_04:0.1,D_05:0.1):0.1):0.1):0.1,"
    "(C_01:0.1,(C_02:0.1,C_03:0.1):0.1):0.2):0.2,"
    "((B_01:0.1,B_02:0.1):0.1,(B_03:0.1,(B_04:0.1,(B_05:0.1,"
    "(B_06:0.1,(B_07:0.1,B_08:0.1):0.1):0.1):0.1):0.1):0.1):0.2,"
    "((C_04:0.1,C_05:0.1):0.1,(C_06:0.1,(D_06:0.1,(D_07:0.1,D_08:0.1):0.1):0.1):0.1):0.2):0.3;"
)

tree = dendropy.Tree.get(data=newick, schema="newick", preserve_underscores=True)

tmpdir = Path(tempfile.mkdtemp(prefix="pyitol_monophyly_"))
tree_path = tmpdir / "virus_tree.nwk"
tree.write_to_path(str(tree_path), schema="newick")

leaf_names = sorted([n.taxon.label for n in tree.leaf_node_iter()])
print(f"Tree written to: {tree_path}")
print(f"Number of leaves: {len(leaf_names)}")
print(f"Leaf names: {leaf_names}")

## 3. Create Virus Taxonomy Metadata

We assign each leaf to a Genotype and Clade based on our designed topology.

根据我们设计的拓扑结构，为每个末端（叶）分配 Genotype 和 Clade。

In [ ]:
# Fix the seed so the simulated collection years are reproducible, in line with
# the deterministic workflow used throughout these tutorials.
random.seed(42)

# Define the genotype assignments
genotype_map = {}
for i in range(1, 9):
    genotype_map[f"A_{i:02d}"] = ("A", "Clade_I")
    genotype_map[f"B_{i:02d}"] = ("B", "Clade_III")
    genotype_map[f"D_{i:02d}"] = ("D", "Clade_II" if i <= 5 else "Clade_IV")
for i in range(1, 7):
    genotype_map[f"C_{i:02d}"] = ("C", "Clade_II" if i <= 3 else "Clade_IV")

records = []
for name in leaf_names:
    geno, clade = genotype_map[name]
    records.append({
        "id": name,
        "Genotype": geno,
        "Clade": clade,
        "Collection_Year": random.choice([2018, 2019, 2020, 2021, 2022]),
    })

taxonomy_df = pd.DataFrame(records)
taxonomy_path = tmpdir / "virus_taxonomy.csv"
taxonomy_df.to_csv(taxonomy_path, index=False)

print(f"Taxonomy saved to: {taxonomy_path}")
taxonomy_df

## 4. Run Monophyly Analysis

We use `check_monophyly()` to test each Genotype group. The function returns a list of
dictionaries with the monophyly status for each group.

我们使用 `check_monophyly()` 检测每个 Genotype 类群的单系性。该函数返回一个字典列表，
其中每一项给出对应类群的单系性状态。

In [ ]:
# Run monophyly check at the Genotype level
results = check_monophyly(
    taxonomy_path=str(taxonomy_path),
    tree_path=str(tree_path),
    rank="Genotype",
    id_column="id",
)

# Display as a DataFrame for clarity
results_df = pd.DataFrame(results)
print("Monophyly analysis results:")
results_df[["group", "status", "lca_node", "member_count", "extra_count", "missing_count"]]

### 4.1 Interpret the Results

Let's examine each group's classification in detail.

我们逐个查看每个类群的判定结果。

In [ ]:
for r in results:
    print(f"\n{'='*60}")
    print(f"Group: {r['group']}")
    print(f"  Status:          {r['status'].upper()}")
    print(f"  LCA node:        {r['lca_node']}")
    print(f"  Members:         {r['member_count']}")
    print(f"  Extra nodes:     {r['extra_count']} ({r['extra_nodes'][:80]}{'...' if len(r['extra_nodes']) > 80 else ''})")
    print(f"  Missing nodes:   {r['missing_count']} ({r['missing_nodes']})")
    if r['subgroups']:
        print(f"  Subgroups:       {r['subgroups'][:120]}{'...' if len(r['subgroups']) > 120 else ''}")

### 4.2 Expected Results Interpretation

Based on our tree construction:

| Group | Expected | Why |
|-------|----------|-----|
| Genotype A | **Monophyletic** | All 8 members in one clade, no outsiders（8 个成员同处一个分支，没有外来成员） |
| Genotype B | **Monophyletic** | All 8 members in one clade, no outsiders（8 个成员同处一个分支，没有外来成员） |
| Genotype C | **Not monophyletic** | Members split across Clade_II (C_01-03) and Clade_IV (C_04-06)（成员被拆分为 Clade_II 中的 C_01-03 与 Clade_IV 中的 C_04-06） |
| Genotype D | **Not monophyletic** | Members split across Clade_II (D_01-05) and Clade_IV (D_06-08)（成员被拆分为 Clade_II 中的 D_01-05 与 Clade_IV 中的 D_06-08） |

根据我们构建树的方式，预期结果如上表所示。请注意：Genotype C 与 D 的成员在根节点的多分叉处
相遇，因此最终的判定标签取决于 `check_monophyly()` 的 `polytomy_mode` 参数如何处理这个
尚未解析的多分叉节点：按 4.1 节调用所用的默认 `polytomy_mode='strict'`，这两个类群都被报告为
`data_insufficient`，而不给出确定的并系/多系判定；若改用 `polytomy_mode='relaxed'`，它们则被
判定为 `paraphyletic`。

The final label therefore depends on how `check_monophyly()` resolves that node through its
`polytomy_mode` argument, because the members of Genotype C and D meet at the unresolved
polytomy at the root: with the default `polytomy_mode='strict'` (the value used by the call
in section 4.1) both groups are reported as `data_insufficient` rather than given a
definitive paraphyletic/polyphyletic call, while `polytomy_mode='relaxed'` classifies them
as `paraphyletic`.

## 5. Monophyly-Based Color-Strip Template

We generate a color-strip template where:
- **Monophyletic** groups get solid, distinct colors
- **Non-monophyletic** groups (paraphyletic or polyphyletic) get a lighter, muted shade

This helps visually identify which groups need taxonomic revision.

我们生成一个按单系性状态着色的色带模板，其中：
- **单系群**使用醒目、彼此区分明显的纯色
- **非单系群**（并系群或多系群）使用更浅、更柔和的色调

这有助于直观地识别出哪些类群需要修订其分类。

In [ ]:
# Assign colors: monophyletic groups get strong colors, polyphyletic get muted
status_color_map = {}
strong_palette = get_nature_palette("vibrant", n=10)
muted_palette = get_nature_palette("pastel", n=10)

genotype_colors = {}
color_idx = 0
for r in results:
    grp = r["group"]
    if r["status"] == "monophyletic":
        genotype_colors[grp] = strong_palette[color_idx % len(strong_palette)]
    else:
        # Polyphyletic/paraphyletic groups get a lighter shade
        genotype_colors[grp] = muted_palette[color_idx % len(muted_palette)]
    color_idx += 1

print("Genotype color mapping (monophyly-based):")
for r in results:
    grp = r["group"]
    status = r["status"]
    color = genotype_colors[grp]
    marker = "*" if status == "monophyletic" else "!"
    print(f"  {marker} {grp:12s} [{status:14s}] -> {color}")

In [ ]:
# Generate the color-strip template
mono_strip_path = tmpdir / "genotype_monophyly_strip.txt"

generate_color_strip_template(
    output_path=mono_strip_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Genotype",
    colors=genotype_colors,
    label="Genotype_monophyly",
    strip_width="60",
)

print(f"Color-strip template written to: {mono_strip_path}")
print("--- Preview ---")
print(mono_strip_path.read_text()[:800])

In [ ]:
# Also generate a branch coloring template
branch_path = tmpdir / "genotype_branch_colors.txt"

generate_branch_template(
    output_path=branch_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Genotype",
    colors=genotype_colors,
    label="Genotype_branches",
)

print(f"Branch template written to: {branch_path}")

## 6. Summary of Generated Files

Let's list all the files generated so far.

列出到目前为止生成的所有文件。

In [ ]:
all_files = sorted(tmpdir.iterdir())
print("Generated files:")
for f in all_files:
    size = f.stat().st_size
    print(f"  {f.name:40s}  {size:>6d} bytes")

## 7. Summary

In this tutorial we demonstrated PyiTOL's monophyly analysis workflow:

1. **Synthetic tree construction**: Built a 30-leaf virus tree with known group structure
2. **Taxonomy metadata**: Created Genotype and Clade columns
3. **Monophyly detection**: Used `check_monophyly()` to classify groups as monophyletic/paraphyletic/polyphyletic
4. **Result interpretation**: Examined extra nodes, missing nodes, and subgroups
5. **Monophyly-based visualization**: Generated color-strip and branch templates with colors reflecting monophyly status

本教程演示了 PyiTOL 的单系性分析工作流：

1. **构建合成树**：构建了一棵具有已知类群结构、包含 30 个末端（叶）的病毒树
2. **分类学元数据**：创建了 Genotype 和 Clade 两列
3. **单系性检测**：使用 `check_monophyly()` 将类群判定为单系群、并系群或多系群
4. **结果解读**：检查了外来节点、缺失节点与亚群信息
5. **基于单系性的可视化**：生成了色带模板与分支模板，其颜色反映各分类群的单系性状态

---

### Key Takeaways / 关键要点

- **Monophyletic groups** are taxonomically coherent; all members share a common ancestor exclusive to the group —— **单系群**在分类学上是一致的：所有成员共享一个仅属于该类群的共同祖先
- **Polyphyletic groups** may need taxonomic revision; members are scattered across the tree —— **多系群**可能需要修订分类：其成员分散在整棵树的不同位置
- The `subgroups` field in the results helps identify how polyphyletic groups are distributed —— 结果中的 `subgroups` 字段有助于判断多系群成员的分布方式
- Using colorblind-friendly palettes ensures accessibility in publications —— 使用对色盲友好的调色板可以保证图表在发表时的可辨识性

---

### Next Steps / 后续步骤

- See `03_batch_workflow.ipynb` for processing multiple trees automatically —— 自动处理多棵树请见 `03_batch_workflow.ipynb`

In [ ]:
# Cleanup (optional)
# import shutil; shutil.rmtree(tmpdir)
print(f"Tutorial complete. Temporary files are in: {tmpdir}")